# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import pandas as pd

# Load your Week-5 model predictions
df_pred = pd.read_csv("../../outputs/model_predictions.csv") # apna file path dalo
df_top20 = df_pred.sort_values('predicted_score', ascending=False).head(20)

# Add reason codes and actions
def get_reason_and_action(row):
    if row['content_type'] == 'video' and row['author_score'] > 0.8:
        return "R1+R2", "PROMOTE: Boost this video immediately. High authority + video format"
    elif row['hashtag_spam_score'] > 0.7:
        return "R3", "REVIEW: Check hashtags. High spam risk despite good score"
    elif row['content_type_new_flag'] == 1:
        return "R4", "TEST SMALL: New content type. Low budget test first"
    else:
        return "R2", "SCHEDULE: Standard publish queue"

df_top20[['reason_code', 'action']] = df_top20.apply(lambda x: pd.Series(get_reason_and_action(x)), axis=1)

# Final Playbook Table
playbook = df_top20[['rank', 'post_id', 'predicted_score', 'reason_code', 'action']]
display(playbook)

playbook.to_csv("../../outputs/action_playbook.csv", index=False)
print("Saved to work/outputs/action_playbook.csv")

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use
Who: SEO team, Content strategists at FlyRank
For what: To prioritize which 20 posts/pages to promote each week for maximum SEO lift. 
Use the ranked queue + reason codes to decide budget and effort.

### Limits - Where this stops being valid
1. **New content types**: Model was trained on image/video/text. If "carousel" or "AI-gen" comes, confidence drops.
2. **Author behavior change**: If author buys followers later, R1+R2 score becomes invalid. Re-run monthly.
3. **Time window**: Scores valid for 30 days only. After that re-train with new data.
4. **Not for**: This does NOT predict virality. Only predicts SEO-relevant engagement based on historical data.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### How to re-run
1. `python scripts/run_model.py --input data/new_week.parquet`
2. `python scripts/generate_playbook.py`
3. Output will be in `work/outputs/action_playbook.csv`

### Human-readable summary
"Do these 5 first: They are R1+R2 and predicted_score > 0.85. 
Avoid these 3: R3 spam risk. Test these 2: New content_type."

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
import pandas as pd
from datetime import datetime

print("### Monitoring / Retrain Triggers ###")

# Load this week's predictions and last week's for comparison
current = pd.read_csv("../../outputs/action_playbook.csv")
last_week = pd.read_csv("../../outputs/action_playbook_lastweek.csv") # isko pehle week me save karna

triggers = []

# Trigger 1: Precision drop
# Assume we have actuals now
if 'actual_lift' in current.columns:
    current_p20 = current.head(20)['actual_lift'].mean()
    if current_p20 < 0.40:
        triggers.append(f"ALERT: Precision@20 dropped to {current_p20:.2f}. Retrain model.")

# Trigger 2: Data drift - new content types
new_content_pct = current['content_type_new_flag'].mean()
if new_content_pct > 0.15:
    triggers.append(f"ALERT: {new_content_pct*100:.1f}% posts are new content_type. Feature update needed.")

# Trigger 3: Prediction distribution shift
score_drop = current['predicted_score'].mean() - last_week['predicted_score'].mean()
if score_drop < -0.10:
    triggers.append(f"ALERT: Avg predicted_score dropped by {abs(score_drop):.2f}. Model drift detected.")

if len(triggers) == 0:
    triggers.append("No triggers fired. Model is healthy this week.")

for t in triggers:
    print(f"- {t} | Checked on: {datetime.now().date()}")

# Save log
log_df = pd.DataFrame({"date": [datetime.now().date()], "triggers": ["; ".join(triggers)]})
log_df.to_csv("../../outputs/monitoring_log.csv", mode='a', header=False, index=False)

**Summary for Paper:**
We monitor 3 things weekly: 
1. **Business Metric**: If Precision@20 < 0.40 for 2 consecutive weeks → Retrain
2. **Data Drift**: If >15% of weekly posts are new `content_type` → Add feature + Retrain  
3. **Prediction Drift**: If average `predicted_score` drops >10% vs last week → Investigate
Logs are appended to `work/outputs/monitoring_log.csv`

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## Self-check

Before you submit, confirm each line honestly:

- [ yes] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks//w07_action_playbook.ipynb` — then submit your repo URL on the card. Done.